<div align="center">

# 🛠️ LangChain Custom Tools

**Three ways to build a custom tool in LangChain**

</div>



### 📚 In this notebook

1. 🎯 Using **`@tool`**
2. 🧱 Using **`StructuredTool`** and **Pydantic**
3. 🏗️ Using the **`BaseTool`** class



### Using @tool Method for bulding the Tool

In [1]:
from langchain_community.tools import tool

/tmp/ipykernel_86433/3868577644.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import tool


In [2]:
# simple function
def multiply(a,b):
    """ Multiply Two Numbers """
    return a*b

# till now it is not the tool but the python doc is important becuase it is desription of function highly recomended for adding so that llm 
# can understand what this function is doing .

In [3]:
# add type hinting highly recomended
def multiply(a:int ,b:int) -> int :
    """ Multiply Two Numbers """
    return a*b


# still not a tool

In [6]:
# now it will become tool becuase @toil decorator make it runnable and package it in a way that llm can interact with this tool.

@tool 
def multiply(a:int ,b:int) -> int :
    """ Multiply Two Numbers """
    return a*b

# now it become a tool

In [7]:
result=multiply.invoke({"a":3,"b":5})

print("Result : ",result)


Result :  15


In [9]:
# now it ihas these things also
print("Description : ",multiply.description)
print("Tool Name : ",multiply.name)
print("Tool Arguments : ",multiply.args)

Description :  Multiply Two Numbers
Tool Name :  multiply
Tool Arguments :  {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [12]:
# The things which are gone to llm instead of body  of function when we attach tool to llm
print("Tool Call : ",multiply.args_schema.model_json_schema())


Tool Call :  {'description': 'Multiply Two Numbers ', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


### using Structured Tool with pydantic 


In [13]:
from langchain_community.tools import StructuredTool
from pydantic import BaseModel,Field

In [14]:
def  multiply(a,b):
    return a*b

In [15]:
class Multiply_ARGS(BaseModel):
    a:int = Field(...,description="First argument to multiply")
    b:int = Field(...,description="Second argument to multiply")

In [22]:
Structured_Multiply_Tool=StructuredTool(
    func=multiply,
    description="Multiply Two Numbers",
    name="Multiply",
    args_schema=Multiply_ARGS
)

In [26]:
result=Structured_Multiply_Tool.invoke({"a":8,"b":3})
print("Result : ",result)

Result :  24


In [27]:
print("Name : ",Structured_Multiply_Tool.name)
print("Description : ",Structured_Multiply_Tool.description)
print("Tool Args : ",Structured_Multiply_Tool.args)

Name :  Multiply
Description :  Multiply Two Numbers
Tool Args :  {'a': {'description': 'First argument to multiply', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'Second argument to multiply', 'title': 'B', 'type': 'integer'}}


### Building the Tools using BaseTool

In [30]:
# BaseTool is the abstract class which give the paritcular interace which every tool class need to implement for built in and custom tools also


from langchain_community.tools import BaseTool
from pydantic import BaseModel, Field
from typing import Type

In [42]:
# Tool Arguments validation Object's Class
class Multiply_Func_Args(BaseModel):
    a: int = Field(...,description="First Argument to multiply")
    b: int = Field(...,description="Second Argument to multiply")


# Tool
class Multiply_Tool(BaseTool):
    name:str="Multiply"
    description:str="Multiply two numbers"

    args_schema:Type[BaseModel]=Multiply_Func_Args

    def _run(self,a: int ,b: int) -> int :
        return a*b

In [43]:
multiply_tool=Multiply_Tool()


result=multiply_tool.invoke({"a":3,"b":3})

print("Result : ",result)

Result :  9


In [44]:
print("Tool Name : ",multiply_tool.name)
print("Tool Description : ",multiply_tool.description)
print("Tool Args : ",multiply_tool.args)

Tool Name :  Multiply
Tool Description :  Multiply two numbers
Tool Args :  {'a': {'description': 'First Argument to multiply', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'Second Argument to multiply', 'title': 'B', 'type': 'integer'}}
